#### LLM Benchmark

##### Importing Libraires and Loading Data

In [1]:
# Libraries; the category data, models and metrics from earlier chunks; AWS credentials from `aws login`

import os
import time

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score

from planscope.category_data import OUT_DIR
from planscope.classifier import CategoryClassifier
from planscope.evaluate import evaluate
from planscope.fill_categories import MODEL_DIR
from planscope.llm_classifier import CATEGORIES, NovaClassifier

os.environ['AWS_PROFILE'] = 'planscope'

In [2]:
# 100 test descriptions per category: every category counts equally, and 900 calls per LLM cost cents

train = pd.read_parquet(OUT_DIR / 'train.parquet')
sample = pd.read_parquet(OUT_DIR / 'test.parquet').groupby('category').sample(100, random_state=42).reset_index(drop=True)
results, timings = {}, {}

In [3]:
# Baseline: the TF-IDF + logistic regression from notebook 02, refitted on the same training texts

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
baseline = LogisticRegression(C=4, max_iter=1000).fit(vectorizer.fit_transform(train['text']), train['category'])
start = time.perf_counter()
results['tf-idf + lr'] = baseline.predict(vectorizer.transform(sample['text']))
timings['tf-idf + lr'] = (time.perf_counter() - start) / len(sample) * 1000

In [4]:
# The fine-tuned DistilBERT exactly as served: int8 ONNX on the CPU, one description at a time

classifier = CategoryClassifier(MODEL_DIR)
start = time.perf_counter()
results['distilbert int8'] = classifier.predict(sample['text'].tolist())[0]
timings['distilbert int8'] = (time.perf_counter() - start) / len(sample) * 1000

In [5]:
# Four LLM set-ups, each paced below its requests-per-minute quota (Pro allows 25, Micro and Lite 200)

examples = train.groupby('category').sample(2, random_state=0)[['text', 'category']].itertuples(index=False)
llms = {
    'nova micro': NovaClassifier('amazon.nova-micro-v1:0', requests_per_minute=150),
    'nova lite': NovaClassifier('amazon.nova-lite-v1:0', requests_per_minute=150),
    'nova pro': NovaClassifier('amazon.nova-pro-v1:0', requests_per_minute=20),
    'nova lite few-shot': NovaClassifier('amazon.nova-lite-v1:0', examples=list(examples), requests_per_minute=150),
}

In [6]:
# Smoke test: 5 descriptions per set-up, to check access and the answer format

{name: [r['label'] for r in llm.classify_all(sample['text'].sample(5, random_state=1))] for name, llm in llms.items()}

{'nova micro': ['trees',
  'trees',
  'trees',
  'householder',
  'lawful_development'],
 'nova lite': ['trees', 'trees', 'trees', 'householder', 'full_planning'],
 'nova pro': ['trees', 'trees', 'trees', 'householder', 'lawful_development'],
 'nova lite few-shot': ['trees',
  'trees',
  'trees',
  'prior_approval',
  'lawful_development']}

In [7]:
# All 900 descriptions cached per set-up

answers = {}
for name, llm in llms.items():
    answers[name] = pd.DataFrame(llm.classify_all(sample['text']))
    results[name] = answers[name]['label'].fillna('invalid')

In [8]:
# Bedrock on-demand prices in US dollars per million tokens (input, output)

PRICES = {
    'amazon.nova-micro-v1:0': (0.035, 0.14),
    'amazon.nova-lite-v1:0': (0.06, 0.24),
    'amazon.nova-pro-v1:0': (0.80, 3.20),
}

In [9]:
# How many of the 900 answers each set-up already has (saved, so a re-run continues from here)

{name: sum(llm.key(text) in llm.cache for text in sample['text']) for name, llm in llms.items()}

{'nova micro': 900,
 'nova lite': 900,
 'nova pro': 900,
 'nova lite few-shot': 900}

##### Results

In [10]:
# Accuracy and macro-F1 on the same 900 descriptions, with 95% bootstrap confidence intervals

pd.concat({name: evaluate(sample['category'], predicted) for name, predicted in results.items()})

score  ci_low  ci_high
tf-idf + lr        accuracy  0.8644  0.8422   0.8878
                   macro_f1  0.8660  0.8457   0.8863
distilbert int8    accuracy  0.8867  0.8667   0.9089
                   macro_f1  0.8882  0.8693   0.9068
nova micro         accuracy  0.6789  0.6500   0.7111
                   macro_f1  0.6797  0.6537   0.7071
nova lite          accuracy  0.7011  0.6700   0.7334
                   macro_f1  0.7119  0.6865   0.7368
nova pro           accuracy  0.7411  0.7111   0.7700
                   macro_f1  0.7394  0.7143   0.7652
nova lite few-shot accuracy  0.7478  0.7189   0.7767
                   macro_f1  0.7550  0.7311   0.7787

In [11]:
# Speed and cost: ms per description, invalid answers, $ per 1,000 descriptions, and $ to label all 278,107 missing ones

MISSING = 278_107  # applications whose category the classifier filled (fill_categories.py)
ops = pd.DataFrame({'ms_per_description': timings})
for name, df in answers.items():
    price_in, price_out = PRICES[llms[name].model_id]
    cost = (df['input_tokens'].sum() * price_in + df['output_tokens'].sum() * price_out) / 1e6
    ops.loc[name, ['ms_per_description', 'invalid_pct', 'usd_per_1000', 'usd_for_missing']] = [
        df['latency_ms'].median(), df['label'].isna().mean() * 100, cost / len(df) * 1000, cost / len(df) * MISSING]
ops.round(4)

,ms_per_description,invalid_pct,usd_per_1000,usd_for_missing
tf-idf + lr,0.0439,NaN,NaN,NaN
distilbert int8,5.2381,NaN,NaN,NaN
nova micro,228.0000,0.0,0.0284,7.9018
nova lite,250.0000,0.0,0.0487,13.5472
nova pro,331.0000,0.0,0.6493,180.5705
nova lite few-shot,270.0000,0.0,0.1095,30.4390


##### Error Analysis

In [12]:
# F1 for each category: where does each model struggle?

pd.DataFrame({name: f1_score(sample['category'], predicted, labels=list(CATEGORIES), average=None)
              for name, predicted in results.items()}, index=list(CATEGORIES)).round(3)

,tf-idf + lr,distilbert int8,nova micro,nova lite,nova pro,nova lite few-shot
householder,0.798,0.796,0.381,0.456,0.562,0.462
full_planning,0.672,0.697,0.473,0.453,0.500,0.443
listed_building,0.639,0.701,0.309,0.346,0.460,0.429
conditions,0.966,0.980,0.880,0.903,0.890,0.881
amendment,0.964,0.980,0.922,0.939,0.910,0.954
lawful_development,0.884,0.911,0.683,0.776,0.783,0.793
prior_approval,0.953,0.958,0.577,0.667,0.646,0.912
trees,0.970,0.990,0.959,0.930,0.965,0.974
advert,0.948,0.980,0.934,0.938,0.938,0.947


In [13]:
# Which categories Nova Pro confuses (rows: truth, columns: prediction; invalid answers are left out, so a row can sum below 1)

labels = list(CATEGORIES)
pd.DataFrame(confusion_matrix(sample['category'], results['nova pro'], labels=labels, normalize='true'),
             index=labels, columns=labels).round(2)

,householder,full_planning,listed_building,conditions,amendment,lawful_development,prior_approval,trees,advert
householder,0.89,0.08,0.01,0.00,0.00,0.00,0.01,0.00,0.01
full_planning,0.36,0.49,0.04,0.01,0.03,0.00,0.02,0.00,0.05
listed_building,0.23,0.19,0.32,0.08,0.06,0.01,0.05,0.00,0.06
conditions,0.00,0.04,0.02,0.89,0.05,0.00,0.00,0.00,0.00
amendment,0.02,0.00,0.00,0.02,0.96,0.00,0.00,0.00,0.00
lawful_development,0.27,0.06,0.00,0.00,0.01,0.65,0.01,0.00,0.00
prior_approval,0.37,0.09,0.00,0.00,0.00,0.00,0.52,0.02,0.00
trees,0.03,0.01,0.00,0.00,0.00,0.00,0.00,0.96,0.00
advert,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.01,0.99


In [14]:
# Descriptions DistilBERT gets right and Nova Pro gets wrong

compare = sample.assign(distilbert=results['distilbert int8'], nova_pro=results['nova pro'])
compare[(compare['distilbert'] == compare['category']) & (compare['nova_pro'] != compare['category'])].head(10)

,text,category,distilbert,nova_pro
16,display of 160 dropped cluster lights across 1...,advert,advert,trees
178,nma - air bricks,amendment,amendment,householder
203,details submitted pursuant to reference 25/021...,conditions,conditions,full_planning
206,minor material amendment to planning applicati...,conditions,conditions,amendment
210,details of energy strategy for residential com...,conditions,conditions,amendment
223,details submitted pursuant to reference 19/033...,conditions,conditions,full_planning
230,submission of amendments to approved details (...,conditions,conditions,amendment
237,details pursuant to ref: 22/02780/lbc for mate...,conditions,conditions,listed_building
241,details submitted pursuant to reference 14/045...,conditions,conditions,full_planning
244,material (section 73a) amendment to planning p...,conditions,conditions,amendment


##### Findings and Decisions
- **Set-up:** 900 test descriptions (100 per category) unseen by every model; labels are the official application types. Nova Micro, Lite and Pro run in-region in London (eu-west-2) via the Bedrock Converse API, temperature 0, forced tool call with an enum of the 9 categories; every answer cached; calls paced under the account quotas (Pro: 25 requests per minute).
- **Accuracy (macro-F1, 95% CI):** TF-IDF + LR 0.866 (0.846–0.886); **DistilBERT int8 0.888 (0.869–0.907)**; Nova Micro 0.680 (0.654–0.707); Nova Lite 0.712 (0.687–0.737); Nova Pro 0.739 (0.714–0.765); Nova Lite few-shot 0.755 (0.731–0.779). DistilBERT's interval does not overlap any LLM's.
- **Structured output:** 0% invalid answers for every Nova set-up.
- **Speed:** TF-IDF 0.04 ms and DistilBERT 5.2 ms per description on a laptop CPU; Nova 228–331 ms median model latency per call (network excluded).
- **Cost:** $0.03 (Micro), $0.05 (Lite), $0.11 (Lite few-shot) and $0.65 (Pro) per 1,000 descriptions; labelling all 278,107 missing applications would cost $8, $14, $30 and $181 (half with batch inference). The local models cost nothing per call. The whole benchmark cost about $0.75.
- **Where the LLMs fail:** legal-route conventions, not language. Nova Pro predicts householder for 36% of full planning, 37% of prior approval and 27% of lawful development applications, and often calls condition discharges ("details submitted pursuant to reference…") full planning. These conventions are learned from 400k labelled examples, not from general knowledge.
- **Few-shot vs size:** two examples per category lifted Lite's prior-approval F1 from 0.667 to 0.912; few-shot Lite matched Pro at about a sixth of the cost.
- **Decision:** keep the fine-tuned DistilBERT for classification. LLMs are the right tool when there is no labelled data (0.68–0.76 macro-F1 with zero training) or to draft labels for a new task.